## Adding a widget to the notebook for specific date

In [0]:
dbutils.widgets.text("ingestion_date", "", "Ingestion Date (YYYY-MM-DD)")
ingestion_date = dbutils.widgets.get("ingestion_date")

if ingestion_date == "":
    from datetime import date
    ingestion_date = str(date.today())

print(f"Validating Bronze ingestion for: {ingestion_date}")

Validating Bronze ingestion for: 2026-08-17


In [0]:
BASE_PATH = "abfss://raw@stmavaluationplatform.dfs.core.windows.net"

sources = {
    "ma_multiples": f"{BASE_PATH}/ma_multiples/ingestion_date={ingestion_date}/multiples.json",
    "ma_multiples_by_year": f"{BASE_PATH}/ma_multiples/ingestion_date={ingestion_date}/multiples-by-year.json"
}

sources

{'ma_multiples': 'abfss://raw@stmavaluationplatform.dfs.core.windows.net/ma_multiples/ingestion_date=2026-08-17/multiples.json',
 'ma_multiples_by_year': 'abfss://raw@stmavaluationplatform.dfs.core.windows.net/ma_multiples/ingestion_date=2026-08-17/multiples-by-year.json'}

### Validation

In [0]:
validation_results = []

for name, path in sources.items():
    try:
        file_info = dbutils.fs.ls(path)
        exists = True
        size_bytes = sum(f.size for f in file_info) if isinstance(file_info, list) else file_info[0].size
    except Exception as e:
        exists = False
        size_bytes = 0
    
    validation_results.append({
        "source_name": name,
        "path": path,
        "file_exists": exists,
        "size_bytes": size_bytes
    })

display(spark.createDataFrame(validation_results))

file_exists,path,size_bytes,source_name
true,abfss://raw@stmavaluationplatform.dfs.core.windows.net/ma_multiples/ingestion_date=2026-08-17/multiples.json,31640,ma_multiples
true,abfss://raw@stmavaluationplatform.dfs.core.windows.net/ma_multiples/ingestion_date=2026-08-17/multiples-by-year.json,50574,ma_multiples_by_year


In [0]:
raw_dataframes = {}

for name, path in sources.items():
    result = next(r for r in validation_results if r["source_name"] == name)
    if not result["file_exists"]:
        print(f"SKIP: {name} — file not found at {path}")
        continue
    
    try:
        df = spark.read.option("multiline", "true").json(path)
        row_count = df.count()
        col_count = len(df.columns)
        raw_dataframes[name] = df
        print(f"{name}: {row_count} rows, {col_count} top-level columns — OK")
    except Exception as e:
        print(f"{name}: FAILED TO PARSE — {str(e)}")

ma_multiples: 1 rows, 7 top-level columns — OK
ma_multiples_by_year: 1 rows, 6 top-level columns — OK


In [0]:
raw_dataframes["ma_multiples"].printSchema()

root
 |-- data: struct (nullable = true)
 |    |-- advertising-agency: struct (nullable = true)
 |    |    |-- 25m_100m_ev: struct (nullable = true)
 |    |    |    |-- ev_revenue: struct (nullable = true)
 |    |    |    |    |-- n: long (nullable = true)
 |    |    |    |    |-- p25: double (nullable = true)
 |    |    |    |    |-- p50: double (nullable = true)
 |    |    |    |    |-- p75: double (nullable = true)
 |    |    |-- 5m_25m_ev: struct (nullable = true)
 |    |    |    |-- ev_revenue: struct (nullable = true)
 |    |    |    |    |-- n: long (nullable = true)
 |    |    |    |    |-- p25: double (nullable = true)
 |    |    |    |    |-- p50: double (nullable = true)
 |    |    |    |    |-- p75: double (nullable = true)
 |    |-- aerospace: struct (nullable = true)
 |    |    |-- 100m_500m_ev: struct (nullable = true)
 |    |    |    |-- ev_revenue: struct (nullable = true)
 |    |    |    |    |-- n: long (nullable = true)
 |    |    |    |    |-- p25: double (nullable

In [0]:
# Capture and display top-level metadata (for Bronze fidelity checks)
multiples_df = raw_dataframes["ma_multiples"]
metadata_row = multiples_df.select("generated_at", "license", "methodology", "min_cell_n", "schema_version", "source_window").first()
print(metadata_row.asDict())

# Count how many sub_verticals are present (sanity check)
sub_verticals = multiples_df.select("data.*").columns
print(f"Sub-verticals found: {len(sub_verticals)}")
print(sub_verticals[:10])

{'generated_at': '2026-08-16T03:30:07Z', 'license': 'MIT (code) / CC-BY 4.0 (data). Cite https://exitvalue.ai if used in publications.', 'methodology': "Aggregate medians/quartiles of disclosed M&A multiples by sub-vertical x EV-bracket. Only post-2018 deals from SEC EDGAR + verified press releases. Cells require n >= 10 per metric. Outliers (EV/EBITDA outside [0.5, 60.0]; EV/Revenue outside [0.05, 25.0]) excluded. Sub-verticals tagged 'other' or 'other-*' excluded.", 'min_cell_n': 10, 'schema_version': '1.0', 'source_window': '2018-2026'}
Sub-verticals found: 46
['advertising-agency', 'aerospace', 'ambulatory-surgery-center', 'apparel', 'auto-dealership', 'auto-parts', 'beverage-manufacturing', 'consulting', 'consumer-products', 'dental-practice']


In [0]:
raw_dataframes["ma_multiples_by_year"].printSchema()


root
 |-- data: struct (nullable = true)
 |    |-- aerospace: struct (nullable = true)
 |    |    |-- 2018: struct (nullable = true)
 |    |    |    |-- ev_revenue: struct (nullable = true)
 |    |    |    |    |-- n: long (nullable = true)
 |    |    |    |    |-- p25: double (nullable = true)
 |    |    |    |    |-- p50: double (nullable = true)
 |    |    |    |    |-- p75: double (nullable = true)
 |    |    |    |-- median_deal_value_m: double (nullable = true)
 |    |    |    |-- n_deals: long (nullable = true)
 |    |    |-- 2019: struct (nullable = true)
 |    |    |    |-- ev_revenue: struct (nullable = true)
 |    |    |    |    |-- n: long (nullable = true)
 |    |    |    |    |-- p25: double (nullable = true)
 |    |    |    |    |-- p50: double (nullable = true)
 |    |    |    |    |-- p75: double (nullable = true)
 |    |    |    |-- median_deal_value_m: double (nullable = true)
 |    |    |    |-- n_deals: long (nullable = true)
 |    |    |-- 2021: struct (nullable =

In [0]:
from datetime import datetime

validation_summary = {
    "ingestion_date": ingestion_date,
    "validated_at": str(datetime.utcnow()),
    "ma_multiples_rows": raw_dataframes["ma_multiples"].count() if "ma_multiples" in raw_dataframes else 0,
    "ma_multiples_by_year_rows": raw_dataframes["ma_multiples_by_year"].count() if "ma_multiples_by_year" in raw_dataframes else 0,
    "sub_verticals_count": len(sub_verticals),
    "all_files_present": all(r["file_exists"] for r in validation_results),
    "status": "PASSED" if all(r["file_exists"] for r in validation_results) else "FAILED"
}

print(validation_summary)

/home/spark-3deaae38-a8f3-4261-980b-52/.ipykernel/1887/command-8680995607945941-571147649:5: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "validated_at": str(datetime.utcnow()),


{'ingestion_date': '2026-08-17', 'validated_at': '2026-08-17 04:56:18.560948', 'ma_multiples_rows': 1, 'ma_multiples_by_year_rows': 1, 'sub_verticals_count': 46, 'all_files_present': True, 'status': 'PASSED'}
